# Convert Update Dataset to GeoJSON

**Author:** NUS PhotoMapper Team  
**Date:** 2026-09-24

## Objective
Convert the updated PhotoMapper Excel dataset to GeoJSON after validating coordinates and basic data quality.

## Inputs
- `data/input/update_dataset_with_disaster_type20260917.xlsx`
- `data/shapefiles/cb_2025_us_state_5m.shp` (and sidecar files, if available)

## Outputs
- `data/geojson_output/update_dataset_with_disaster_type20260917.geojson`
- `data/geojson_output/update_dataset_with_disaster_type20260917_whole_corrected.geojson`
- `data/geojson_output/update_dataset_with_disaster_type20260917_USA_clean.geojson`
- `data/geojson_output/update_dataset_with_disaster_type20260917_outside_US_review.geojson`
- `data/geojson_output/us_states_2025.geojson`

In [ ]:
# 1. Imports and configuration

from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd()
INPUT_FILE = PROJECT_DIR / "update_dataset_with_disaster_type20260917.xlsx"
OUTPUT_DIR = PROJECT_DIR / "geojson_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "update_dataset_with_disaster_type20260917.geojson"

CRS_WGS84 = "EPSG:4326"

print(f"Input file: {INPUT_FILE}")
print(f"Output file: {OUTPUT_FILE}")


In [ ]:
# 2. Load, validate, and convert to GeoJSON

df = pd.read_excel(INPUT_FILE)
required_columns = ["X", "Y"]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise KeyError(f"Missing required coordinate columns: {missing_columns}")

df = df.copy()
df["longitude"] = pd.to_numeric(df["X"], errors="coerce")
df["latitude"] = pd.to_numeric(df["Y"], errors="coerce")

valid_mask = df["longitude"].between(-180, 180) & df["latitude"].between(-90, 90)
invalid_count = int((~valid_mask).sum())
if invalid_count:
    print(f"Dropping {invalid_count:,} rows with invalid coordinates.")

valid_df = df.loc[valid_mask].copy()
gdf = gpd.GeoDataFrame(
    valid_df,
    geometry=gpd.points_from_xy(valid_df["longitude"], valid_df["latitude"]),
    crs=CRS_WGS84,
)

# Validate that the coordinate columns behave like lon/lat pairs.
lon_like = gdf["longitude"].between(-180, 180).all()
lat_like = gdf["latitude"].between(-90, 90).all()
print(f"Longitude valid: {lon_like}")
print(f"Latitude valid: {lat_like}")
print(f"Records retained: {len(gdf):,}")

gdf.to_file(OUTPUT_FILE, driver="GeoJSON")
print("GeoJSON saved.")

In [ ]:
# 3. Quick preview

preview_gdf = gpd.read_file(OUTPUT_FILE)
print(f"Preview rows: {len(preview_gdf):,}")
print(f"CRS: {preview_gdf.crs}")
print("Total bounds [minx, miny, maxx, maxy]:", preview_gdf.total_bounds)

preview_columns = [column for column in ["X", "Y", "longitude", "latitude", "Disaster Type", "CreationDate"] if column in preview_gdf.columns]
display(preview_gdf[preview_columns].head(10))

print("First GeoJSON feature preview:")
preview_json_frame = preview_gdf.head(1).copy()
for column in preview_json_frame.select_dtypes(include=["datetime64[ns]", "datetimetz"]).columns:
    preview_json_frame[column] = preview_json_frame[column].astype(str)
print(preview_json_frame.to_json())


In [ ]:
# 4. Map preview

# Reuse the exported GeoJSON so the preview matches the saved output exactly.
map_preview_gdf = gpd.read_file(OUTPUT_FILE)

# A light-weight preview: sample points for readability, but keep the full geographic extent.
preview_sample = map_preview_gdf.sample(n=min(5000, len(map_preview_gdf)), random_state=42).copy()

fig, ax = plt.subplots(figsize=(12, 7))
map_preview_gdf.total_bounds
preview_sample.plot(ax=ax, color="#b22222", markersize=2, alpha=0.45)
ax.set_title("GeoJSON Map Preview", fontsize=14, fontweight="bold")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xlim(map_preview_gdf.total_bounds[0], map_preview_gdf.total_bounds[2])
ax.set_ylim(map_preview_gdf.total_bounds[1], map_preview_gdf.total_bounds[3])
ax.grid(True, color="#dddddd", linewidth=0.5)
plt.show()


In [ ]:
# 5. Diagnose non-USA anomalies and apply a conservative fix

USA_BOXES = {
    "CONUS": (-125.0, -66.0, 24.0, 50.0),
    "Alaska": (-180.0, -129.0, 51.0, 72.0),
    "Hawaii": (-161.0, -154.0, 18.0, 23.0),
    "PuertoRico_USVI": (-68.5, -64.0, 17.0, 19.0),
    "Guam_CNMI": (144.0, 146.8, 13.0, 21.0),
    "AmericanSamoa": (-171.5, -168.5, -15.5, -10.0),
}


def assign_us_region(frame, lon_col="longitude", lat_col="latitude"):
    region = pd.Series(pd.NA, index=frame.index, dtype="object")
    for name, (xmin, xmax, ymin, ymax) in USA_BOXES.items():
        mask = frame[lon_col].between(xmin, xmax) & frame[lat_col].between(ymin, ymax)
        region.loc[mask] = name
    return region


audit_df = gdf.copy()
audit_df["us_region"] = assign_us_region(audit_df)
outside_mask = audit_df["us_region"].isna()
outside_df = audit_df.loc[outside_mask].copy()

# Swap check: rows that would land in a US region if lon/lat were swapped.
outside_df["swap_region_candidate"] = assign_us_region(
    outside_df.rename(columns={"longitude": "latitude", "latitude": "longitude"}),
    lon_col="longitude",
    lat_col="latitude",
)
outside_df["swap_to_us"] = outside_df["swap_region_candidate"].notna()

print(f"Rows in dataset: {len(audit_df):,}")
print(f"Rows outside US boxes: {outside_mask.sum():,}")
print(f"Outside rows likely caused by swapped lon/lat: {outside_df['swap_to_us'].sum():,}")
if len(outside_df):
    print(f"Swap-like share of outside rows: {outside_df['swap_to_us'].mean() * 100:.2f}%")

print("\nTop outside clusters by 10-degree bins (lon, lat):")
cluster_table = (
    outside_df.assign(
        lon_bin10=(outside_df["longitude"] // 10 * 10).astype(int),
        lat_bin10=(outside_df["latitude"] // 10 * 10).astype(int),
    )
    .groupby(["lon_bin10", "lat_bin10"])
    .size()
    .sort_values(ascending=False)
    .head(10)
)
display(cluster_table.to_frame("count"))

sample_cols = [col for col in ["CreationDate", "Disaster Type", "Incident", "Location_Name", "X", "Y", "longitude", "latitude", "swap_to_us"] if col in outside_df.columns]
print("\nSample outside rows for manual QA:")
display(outside_df[sample_cols].head(12))

# Conservative fix: only swap coordinates for rows that are outside US boxes but map into US after swapping.
corrected_df = audit_df.copy()
corrected_df["coordinate_swap_applied"] = False
swap_idx = outside_df.index[outside_df["swap_to_us"]]
orig_lon = corrected_df.loc[swap_idx, "longitude"].copy()
corrected_df.loc[swap_idx, "longitude"] = corrected_df.loc[swap_idx, "latitude"].values
corrected_df.loc[swap_idx, "latitude"] = orig_lon.values
corrected_df.loc[swap_idx, "coordinate_swap_applied"] = True
corrected_df = gpd.GeoDataFrame(
    corrected_df,
    geometry=gpd.points_from_xy(corrected_df["longitude"], corrected_df["latitude"]),
    crs=CRS_WGS84,
)

corrected_df["us_region"] = assign_us_region(corrected_df)
usa_clean_gdf = corrected_df.loc[corrected_df["us_region"].notna()].copy()
foreign_review_gdf = corrected_df.loc[corrected_df["us_region"].isna()].copy()

WHOLE_CORRECTED_FILE = OUTPUT_DIR / "update_dataset_with_disaster_type20260917_whole_corrected.geojson"
USA_CLEAN_FILE = OUTPUT_DIR / "update_dataset_with_disaster_type20260917_USA_clean.geojson"
FOREIGN_REVIEW_FILE = OUTPUT_DIR / "update_dataset_with_disaster_type20260917_outside_US_review.geojson"

corrected_df.to_file(WHOLE_CORRECTED_FILE, driver="GeoJSON")
usa_clean_gdf.to_file(USA_CLEAN_FILE, driver="GeoJSON")
foreign_review_gdf.to_file(FOREIGN_REVIEW_FILE, driver="GeoJSON")

print("\nSuggested fix outputs:")
print(f"Whole corrected GeoJSON: {WHOLE_CORRECTED_FILE}")
print(f"USA-clean GeoJSON: {USA_CLEAN_FILE}")
print(f"Outside-US review GeoJSON: {FOREIGN_REVIEW_FILE}")
print(f"Whole corrected rows: {len(corrected_df):,}")
print(f"USA-clean rows: {len(usa_clean_gdf):,}")
print(f"Outside-US rows after swap-fix: {len(foreign_review_gdf):,}")
print(f"Coordinate swaps applied: {int(corrected_df['coordinate_swap_applied'].sum()):,}")


In [ ]:
# 6. Optional: build US states GeoJSON (auto-download if local shapefile is missing)

from pathlib import Path
import urllib.request
import zipfile

import geopandas as gpd

states_shp = PROJECT_DIR / "data" / "shapefiles" / "cb_2025_us_state_5m.shp"
states_zip = PROJECT_DIR / "data" / "shapefiles" / "cb_2025_us_state_5m.zip"

if not states_shp.exists():
    census_zip_url = "https://www2.census.gov/geo/tiger/GENZ2025/shp/cb_2025_us_state_5m.zip"
    print(f"State shapefile not found. Downloading: {census_zip_url}")
    urllib.request.urlretrieve(census_zip_url, states_zip)
    with zipfile.ZipFile(states_zip, "r") as zf:
        zf.extractall(PROJECT_DIR / "data" / "shapefiles")

states = gpd.read_file(states_shp).to_crs("EPSG:4326")
states_output = OUTPUT_DIR / "us_states_2025.geojson"
states.to_file(states_output, driver="GeoJSON")
print(f"Saved: {states_output}")
print(f"State features: {len(states):,}")